# 03. Threshold Sweep

**Question**: agent-kms's README claims `score_threshold=0.93` is the
sweet spot for `multilingual-e5-base` on Japanese-heavy corpora. Does
that hold on this **public-documentation** corpus?

We sweep T from 0.78 to 0.94 in 0.01 steps and compute per-query
Precision, Recall, F1, MRR, NDCG@10 plus `num_returned`, then average
across queries (excluding adversarial / empty-gold queries from
the precision/recall aggregate — adversarial queries are scored
separately as **empty-rate**).


In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from src.eval_utils import resolve_eval_paths

CORPUS_DIR, QUERIES_PATH, COLLECTION = resolve_eval_paths(ROOT)
os.environ["QDRANT_URL"] = "http://localhost:6433"
os.environ["AGENT_KMS_COLLECTION"] = COLLECTION
os.environ["AGENT_KMS_CONFIG"] = str(ROOT / ".agent-kms" / "kms.toml")
os.environ["AGENT_KMS_KNOWLEDGE_ROOT"] = str(CORPUS_DIR)

from agent_kms.retrieve import retrieve
from src.eval_utils import load_queries, sweep_threshold, gold_set, chunk_key

sns.set_theme(style="whitegrid")


In [ ]:
queries = load_queries(QUERIES_PATH)
in_domain   = [q for q in queries if q.get("gold")]
adversarial = [q for q in queries if not q.get("gold")]
print(f"{len(queries)} total queries  |  in-domain {len(in_domain)}  |  adversarial {len(adversarial)}")

thresholds = np.round(np.arange(0.78, 0.945, 0.01), 3).tolist()
print(f"sweeping T over {len(thresholds)} steps: {thresholds[0]} -> {thresholds[-1]}")


In [ ]:
def retrieve_fn(q, score_threshold):
    return retrieve(q, score_threshold=score_threshold)

rows = sweep_threshold(in_domain, retrieve_fn, thresholds)
df_in = pd.DataFrame(rows)
print(f"in-domain sweep rows: {len(df_in)}")

# Adversarial — track only num_returned (gold is empty so P/R undefined)
adv_rows = []
for t in thresholds:
    for q in adversarial:
        r = retrieve(q["query"], score_threshold=t)
        adv_rows.append({"threshold": t, "query_id": q["id"], "num_returned": len(r)})
df_adv = pd.DataFrame(adv_rows)
print(f"adversarial rows: {len(df_adv)}")


In [ ]:
agg = df_in.groupby("threshold").agg(
    precision=("precision", "mean"),
    recall=("recall", "mean"),
    f1=("f1", "mean"),
    mrr=("mrr", "mean"),
    ndcg10=("ndcg10", "mean"),
    avg_returned=("num_returned", "mean"),
).reset_index()

# empty_rate_adversarial = fraction of adversarial queries that returned 0
empty_adv = df_adv.groupby("threshold")["num_returned"].apply(lambda s: (s == 0).mean()).rename("adv_empty_rate")
agg = agg.merge(empty_adv, on="threshold")

# empty_rate_indomain = fraction of in-domain queries that returned 0 (= total recall miss)
empty_in = df_in.groupby("threshold")["num_returned"].apply(lambda s: (s == 0).mean()).rename("indomain_empty_rate")
agg = agg.merge(empty_in, on="threshold")

agg.round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
for col, color in [("precision", "#1f77b4"), ("recall", "#ff7f0e"), ("f1", "#2ca02c")]:
    ax.plot(agg["threshold"], agg[col], marker="o", label=col, color=color)
ax.axvline(0.93, ls="--", color="grey", alpha=0.6, label="agent-kms default (0.93)")
best_t = agg.loc[agg["f1"].idxmax(), "threshold"]
ax.axvline(best_t, ls=":", color="red", alpha=0.6, label=f"best F1 @ T={best_t}")
ax.set_xlabel("score_threshold")
ax.set_ylabel("metric value (mean over in-domain queries)")
ax.set_title("Precision / Recall / F1 vs threshold")
ax.legend()

ax = axes[1]
ax.plot(agg["threshold"], agg["avg_returned"], marker="o", color="#9467bd", label="avg returned (in-domain)")
ax2 = ax.twinx()
ax2.plot(agg["threshold"], agg["adv_empty_rate"], marker="s", color="#d62728", label="adversarial empty-rate")
ax2.plot(agg["threshold"], agg["indomain_empty_rate"], marker="x", color="#8c564b", label="in-domain empty-rate")
ax.set_xlabel("score_threshold")
ax.set_ylabel("avg returned chunks", color="#9467bd")
ax2.set_ylabel("empty-rate", color="#d62728")
ax2.set_ylim(0, 1.05)
ax.set_title("Volume + empty-rate vs threshold")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig(ROOT / "results" / "fig_threshold_sweep.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Best F1 + a 'budgeted' best (constrain volume so a downstream LLM
# doesn't blow its context window). We use avg_returned <= 10 as the budget.
best_row = agg.sort_values("f1", ascending=False).iloc[0]
print(f"Best F1 (unconstrained): T={best_row['threshold']:.2f}  "
      f"P={best_row['precision']:.3f} R={best_row['recall']:.3f} F1={best_row['f1']:.3f}  "
      f"avg_returned={best_row['avg_returned']:.1f}")

budget = agg[agg["avg_returned"] <= 10]
if len(budget):
    b = budget.sort_values("f1", ascending=False).iloc[0]
    print(f"Best F1 (avg_returned <= 10): T={b['threshold']:.2f}  "
          f"P={b['precision']:.3f} R={b['recall']:.3f} F1={b['f1']:.3f}  "
          f"avg_returned={b['avg_returned']:.1f}")

# Save the table
agg.to_csv(ROOT / "results" / "threshold_sweep.csv", index=False)
print("\nsaved results/threshold_sweep.csv")


## Findings

Empirical sweep (multilingual-e5-base, all retrieve enhancements on):

| threshold | precision | recall | F1    | avg returned | adv empty | in-domain empty |
|-----------|-----------|--------|-------|--------------|-----------|-----------------|
| 0.78      | 0.026     | 1.000  | 0.049 | 81.6         | 0.00      | 0.00            |
| 0.80      | 0.076     | 0.756  | 0.126 | 29.7         | 0.25      | 0.04            |
| **0.81**  | **0.159** | **0.649** | **0.201** | **13.2** | **0.75** | **0.04**     |
| 0.82      | 0.163     | 0.423  | 0.182 | 5.3          | 1.00      | 0.21            |
| 0.83      | 0.152     | 0.280  | 0.173 | 1.8          | 1.00      | 0.57            |
| 0.85      | 0.077     | 0.107  | 0.086 | 0.4          | 1.00      | 0.82            |
| 0.87+     | 0         | 0      | 0     | 0            | 1.00      | 1.00            |
| **0.93**  | **0**     | **0**  | **0** | **0**        | **1.00**  | **1.00**        |

- The README's default **T=0.93 is structurally unusable on this corpus**
  — every query returns zero chunks. The "sweet spot" the README reports
  holds on its **original corpus** (Anthropic-style internal knowledge
  with curated severity tags) but does not transfer to public Japanese
  technical documentation.
- **Best F1 on this corpus is T = 0.81** (F1 = 0.201) with `e5-base`.
  Precision is intrinsically limited at this threshold (~0.16) because
  agent-kms is threshold-based, not top-k: every chunk that crosses the
  bar returns, so a query whose gold has 2 chunks but 11 chunks cross
  the bar gets P ≈ 0.18 even when both gold items are retrieved.
  Recall (0.65) and MRR (0.47) are the metrics that better reflect
  retrieval quality at this regime.
- The **adversarial empty-rate jumps from 0 to 1.0 between T=0.81 and
  T=0.82** — there is a sharp boundary where out-of-domain queries stop
  returning anything. T=0.82-0.83 is the conservative choice if false
  positives are costly (e.g. LLM hallucination risk).
- The Japanese-on-Japanese baseline cosine sits high (~0.80 floor even
  for unrelated content), so the usable discrimination band is narrow
  (≈ 0.78–0.86 here, just 0.08 wide).

**Implication**: agent-kms's threshold default should be **corpus-tuned**,
not a global constant. The CLI / config layer already exposes
`score_threshold` as a tunable, so the fix is documentation +
a small calibration recipe, not a code change.
